## Download model-folder from huggingface

In [ ]:
from huggingface_hub import snapshot_download
"""
snapshot_download(
    repo_id="microsoft/Phi-4-mini-instruct",
    local_dir="../models/phi-4-mini-instruct-src",
    local_dir_use_symlinks=False,
)
"""

## Convert to OPenVINO format (for NPU compatability)

In [ ]:
# Local-to-OpenVINO conversion (no model download in code)
# Input: local HF model folder with config + safetensors
# Output: OpenVINO model folder

from pathlib import Path
from optimum.intel.openvino import OVModelForCausalLM
from transformers import AutoTokenizer
"""
local_src = Path("../models/phi-4-mini-instruct-src")   # local source model folder
ov_out = Path("../models/phi4-ov-fp16-local")           # output folder
ov_out.mkdir(parents=True, exist_ok=True)

# Load tokenizer locally
tokenizer = AutoTokenizer.from_pretrained(local_src, local_files_only=True)

# Convert from local model files to OpenVINO representation
model = OVModelForCausalLM.from_pretrained(
    local_src.as_posix(),
    export=True,
    compile=False,
    local_files_only=True
)

# Save OpenVINO artifacts locally
model.save_pretrained(ov_out.as_posix())
tokenizer.save_pretrained(ov_out.as_posix())

print("Saved OpenVINO model to:", ov_out.resolve())
print("Files:", [p.name for p in ov_out.iterdir()])
"""

c:\Users\matse\GIG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\matse\GIG\.venv\Lib\site-packages\optimum\intel\openvino\modeling_base.py:602: SyntaxWarning: invalid escape sequence '\.'
  pattern=cls._search_pattern if not kwargs.get("from_onnx", False) else ".*\.onnx$",
Could not infer whether the model was already converted or not to the OpenVINO IR, keeping `export=True`.
[Errno 2] No such file or directory: 'C:\\Users\\matse\\.cache\\huggingface\\hub\\models--..--models--phi-4-mini-instruct-src\\refs\\main'
`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00,  5.21it/s]
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.
c:\Users\matse\GIG\.venv\Lib\site-packages\transformers\cach

Saved OpenVINO model to: C:\Users\matse\GIG\models\phi4-ov-fp16-local
Files: ['added_tokens.json', 'chat_template.jinja', 'config.json', 'generation_config.json', 'merges.txt', 'openvino_model.bin', 'openvino_model.xml', 'special_tokens_map.json', 'tokenizer.json', 'tokenizer_config.json', 'vocab.json']


In [5]:
from openvino import Core
core = Core()
m = core.read_model("../models/phi4-ov-fp16-local/openvino_model.xml")
core.compile_model(m, "CPU")
core.compile_model(m, "GPU")
print("IR compiles on CPU/GPU")

IR compiles on CPU/GPU


## quantisize model

In [ ]:
from __future__ import annotations

import subprocess
from pathlib import Path


def run(cmd: list[str]) -> None:
    print(">", " ".join(cmd))
    p = subprocess.run(cmd, text=True, capture_output=True)
    print("---- STDOUT ----")
    print(p.stdout)
    print("---- STDERR ----")
    print(p.stderr)
    if p.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {p.returncode}")


def export_quantized_openvino(
    src: str | Path,
    out: str | Path,
    bits: int = 8,
    task: str = "text-generation-with-past",
) -> Path:
    """Export a local HF model folder to quantized OpenVINO artifacts.

    Args:
        src: Local source model folder (config + tokenizer + safetensors).
        out: Output folder for OpenVINO model files.
        bits: Quantization bit-width, either 8 or 4.
        task: Optimum export task.

    Returns:
        Resolved output path.
    """
    src_path = Path(src).resolve()
    out_path = Path(out).resolve()
    out_path.mkdir(parents=True, exist_ok=True)

    if not src_path.exists():
        raise FileNotFoundError(f"Source model folder not found: {src_path}")
    if bits not in (8, 4):
        raise ValueError("bits must be 8 or 4")

    weight_format = "int8" if bits == 8 else "int4"

    cmd = [
        "optimum-cli",
        "export",
        "openvino",
        "--model",
        str(src_path),
        "--task",
        task,
        "--weight-format",
        weight_format,
        str(out_path),
    ]
    run(cmd)

    print("\nDone.")
    print(f"Source: {src_path}")
    print(f"Output: {out_path}")
    print("Expected files include: openvino_model.xml, openvino_model.bin, tokenizer files")
    return out_path


# Example usage from notebook/script:
# export_quantized_openvino(
#     src="../models/phi-4-mini-instruct-src",
#     out="../models/phi4-ov-int8-local",
#     bits=8,
# )

export_quantized_openvino(
    src="../models/phi-4-mini-instruct-src",
    out="../models/phi4-ov-int8-local",
    bits=8
    )

> optimum-cli export openvino --model C:\Users\matse\GIG\models\phi-4-mini-instruct-src --task text-generation --weight-format int8 C:\Users\matse\GIG\models\phi4-ov-int8-local
---- STDOUT ----
INFO:nncf:Statistics of the bitwidth distribution:
+---------------------------+-----------------------------+----------------------------------------+
| Weight compression mode   | % all parameters (layers)   | % ratio-defining parameters (layers)   |
+===========================+=============================+========================================+
| int8_asym, per-channel    | 100% (129 / 129)            | 100% (129 / 129)                       |
+---------------------------+-----------------------------+----------------------------------------+
Applying Weight Compression --------------------------   0% â€¢ 0:00:00 â€¢ -:--:--
Applying Weight Compression --------------------------   0% â€¢ 0:00:00 â€¢ -:--:--
Applying Weight Compression --------------------------   0% â€¢ 0:00:00 â€¢ -:--:-

RuntimeError: Command failed with exit code 1

In [3]:
from openvino import Core

core = Core()
model_path = "../models/phi4-ov-int8-local/openvino_model.xml"
model = core.read_model(model_path)

print("Inputs before reshape:")
for i in model.inputs:
    print(i.get_any_name(), i.get_partial_shape())

Inputs before reshape:
input_ids [?,?]
attention_mask [?,?]
position_ids [?,?]
beam_idx [?]


In [4]:
from openvino import Core, PartialShape

core = Core()
model = core.read_model("../models/phi4-ov-int8-local/openvino_model.xml")

new_shapes = {}
for inp in model.inputs:
    ps = inp.get_partial_shape()
    static = []
    for d in ps:
        static.append(1 if d.is_dynamic else int(d.get_length()))
    new_shapes[inp.get_any_name()] = PartialShape(static)

model.reshape(new_shapes)

compiled = core.compile_model(model, "NPU")
print("Compiled on:", compiled.get_property("EXECUTION_DEVICES"))

RuntimeError: Exception from src\inference\src\cpp\core.cpp:113:
Exception from src\inference\src\dev\plugin.cpp:53:
Exception from src\plugins\intel_npu\src\plugin\src\plugin.cpp:879:
Exception from src\plugins\intel_npu\src\compiler_adapter\src\ze_graph_ext_wrappers.cpp:405:
L0 pfnCreate2 result: ZE_RESULT_ERROR_INVALID_ARGUMENT, code 0x78000004 - generic error code for invalid arguments . [NPU_VCL] Compiler returned msg:
Exception from src\core\src\partial_shape.cpp:266:
to_shape was called on a dynamic shape.






In [2]:
# Verify that the model loads and compiles on NPU
from openvino import Core
model_path = "../models/phi4-ov-int8-local/openvino_model.xml"
core = Core()
model = core.read_model(model_path)
compiled = core.compile_model(model, "NPU")
print("Compiled on:", compiled.get_property("EXECUTION_DEVICES"))

RuntimeError: Exception from src\inference\src\cpp\core.cpp:113:
Exception from src\inference\src\dev\plugin.cpp:53:
Exception from src\plugins\intel_npu\src\plugin\src\plugin.cpp:879:
Exception from src\plugins\intel_npu\src\compiler_adapter\src\ze_graph_ext_wrappers.cpp:405:
L0 pfnCreate2 result: ZE_RESULT_ERROR_INVALID_ARGUMENT, code 0x78000004 - generic error code for invalid arguments . [NPU_VCL] Compiler returned msg:
Exception from src\core\src\partial_shape.cpp:266:
to_shape was called on a dynamic shape.






## minimal setup for running PHi-4-mini-instruct locally on NPU

In [1]:
import onnxruntime as ort
print(ort.get_available_providers())

['OpenVINOExecutionProvider', 'CPUExecutionProvider']


In [3]:
import onnxruntime as ort
from transformers import PreTrainedTokenizerFast
import numpy as np


# -----------------------------
# 1. Load tokenizer locally
# -----------------------------
tokenizer = PreTrainedTokenizerFast.from_pretrained(
    "../models/phi-4-mini-instruct-onnx",  # folder containing tokenizer.json etc.
    local_files_only=True
)

eos_token_ids = tokenizer.eos_token_id
if isinstance(eos_token_ids, int):
    eos_token_ids = {eos_token_ids}
else:
    eos_token_ids = set(eos_token_ids)

# -----------------------------
# 2. Load ONNX model on NPU
# -----------------------------
session = ort.InferenceSession(
    "../models/phi-4-mini-instruct-onnx/model.onnx",
    providers=[("OpenVINOExecutionProvider", {"device_type": "NPU"})]
)

#print("Providers:", session.get_providers())
#print("Provider options:", session.get_provider_options())

input_metas = session.get_inputs()
output_metas = session.get_outputs()
input_names = [i.name for i in input_metas]
output_names = [o.name for o in output_metas]
past_input_names = [n for n in input_names if n.startswith("past_key_values.")]

#print(f"Loaded model with {len(input_names)} inputs and {len(output_names)} outputs")
#print("First inputs:", input_names[:6], "...")


def ort_type_to_np_dtype(ort_type: str):
    mapping = {
        "tensor(float)": np.float32,
        "tensor(float16)": np.float16,
        "tensor(int64)": np.int64,
        "tensor(int32)": np.int32,
        "tensor(bool)": np.bool_,
    }
    if ort_type not in mapping:
        raise ValueError(f"Unsupported ONNX input type: {ort_type}")
    return mapping[ort_type]


def _resolve_dynamic_dim(dim_name: str, axis: int, batch_size: int, past_seq_len: int):
    name = str(dim_name).lower()
    if "batch" in name:
        return batch_size
    if "past" in name:
        return past_seq_len
    if "seq" in name or "sequence" in name:
        return past_seq_len if axis >= 2 else 1
    return 1


def init_past_cache(batch_size: int, past_seq_len: int = 0):
    past = {}
    for meta in input_metas:
        if not meta.name.startswith("past_key_values."):
            continue

        shape = []
        for axis, d in enumerate(meta.shape):
            if isinstance(d, int):
                shape.append(d)
            else:
                shape.append(_resolve_dynamic_dim(d, axis, batch_size, past_seq_len))

        dtype = ort_type_to_np_dtype(meta.type)
        past[meta.name] = np.zeros(shape, dtype=dtype)

    return past


def extract_next_past_from_outputs(outputs):
    next_past = {}
    for name, value in zip(output_names, outputs):
        if name.startswith("present."):
            next_past[name.replace("present.", "past_key_values.")] = value

    missing = [n for n in past_input_names if n not in next_past]
    if missing:
        raise ValueError(f"Model did not return all present cache tensors. Missing: {missing[:4]}...")

    return next_past


def run_step(input_ids_step, past):
    batch_size = input_ids_step.shape[0]
    any_cache = next(iter(past.values()))
    past_seq_len = any_cache.shape[2]

    # For this export, total_sequence_length = past_sequence_length + current step length.
    attention_mask = np.ones((batch_size, past_seq_len + input_ids_step.shape[1]), dtype=np.int64)

    feed = {
        "input_ids": input_ids_step,
        "attention_mask": attention_mask,
    }
    feed.update(past)

    outputs = session.run(None, feed)
    logits = outputs[0]
    next_past = extract_next_past_from_outputs(outputs)
    return logits, next_past


def generate(prompt, max_new_tokens=100):
    prompt_ids = tokenizer.encode(prompt, return_tensors="np").astype(np.int64)
    batch_size = prompt_ids.shape[0]
    prompt_len = prompt_ids.shape[1]

    generated_ids = prompt_ids.copy()
    past = init_past_cache(batch_size=batch_size, past_seq_len=0)

    # Token-by-token prefill is more robust with DML GroupQueryAttention than a single long prefill.
    last_logits = None
    for pos in range(prompt_len):
        input_step = prompt_ids[:, pos:pos + 1]
        last_logits, past = run_step(input_step, past)

    for _ in range(max_new_tokens):
        next_token_id = int(np.argmax(last_logits[0, -1]))
        if next_token_id in eos_token_ids:
            break

        next_token = np.array([[next_token_id]], dtype=np.int64)
        generated_ids = np.concatenate([generated_ids, next_token], axis=1)

        last_logits, past = run_step(next_token, past)

    return tokenizer.decode(generated_ids[0], skip_special_tokens=True)


# -----------------------------
# 4. Try it
# -----------------------------
#prompt = "Explain how NPUs accelerate transformer models."
prompt = "your task is to extract classes from the following sentence, to be used in an OWL 2 language ontology: 'SENS is an integrated system for collecting physical activity data from groups of people.' List the classes and their relationships."
output = generate(prompt, max_new_tokens=100)

print("OUTPUT: ", output)

The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'GPT2Tokenizer'. 
The class this function is called from is 'PreTrainedTokenizerFast'.


OUTPUT:  your task is to extract classes from the following sentence, to be used in an OWL 2 language ontology: 'SENS is an integrated system for collecting physical activity data from groups of people.' List the classes and their relationships. Here is the list of classes and their relationships:
1. SENS (Class)
2. Physical Activity Data (Class)
3. Group of People (Class)
4. Integrated System (Class)

Relationships:
1. SENS is an Integrated System (SENS - Integrated System)
2. Integrated System collects Physical Activity Data (Integrated System - Physical Activity Data)
3. Integrated System collects data from Group of People (Integrated System - Group of People)


In [7]:
import onnxruntime as ort
import json
import os

so = ort.SessionOptions()
so.enable_profiling = True

session = ort.InferenceSession(
    "../models/phi-4-mini-instruct/model.onnx",
    sess_options=so,
    providers=[("OpenVINOExecutionProvider", {"device_type": "NPU"})]
)

print("Providers:", session.get_providers())
print("Provider options:", session.get_provider_options())

# Run enough work to make utilization visible
for _ in range(5):
    _ = generate("Explain NPUs in one paragraph.", max_new_tokens=200)

profile_path = session.end_profiling()
print("Profile:", profile_path)

with open(profile_path, "r", encoding="utf-8") as f:
    trace = json.load(f)

providers = sorted({
    e.get("args", {}).get("provider")
    for e in trace
    if isinstance(e, dict) and "args" in e and isinstance(e["args"], dict) and "provider" in e["args"]
})
print("Providers seen in trace:", providers)

Providers: ['CPUExecutionProvider']
Provider options: {'CPUExecutionProvider': {}}
Profile: onnxruntime_profile__2026-03-09_15-58-16.json
Providers seen in trace: ['CPUExecutionProvider']


In [9]:
import onnxruntime as ort

so = ort.SessionOptions()
so.add_session_config_entry("session.disable_cpu_ep_fallback", "1")

session = ort.InferenceSession(
    "../models/phi-4-mini-instruct/model.onnx",
    sess_options=so,
    providers=[("OpenVINOExecutionProvider", {"device_type": "NPU"})],
)

print(session.get_providers())
print(session.get_provider_options())

Fail: [ONNXRuntimeError] : 1 : FAIL : This session contains graph nodes that are assigned to the default CPU EP, but fallback to CPU EP has been explicitly disabled by the user.

In [4]:


from openvino import Core
core = Core()
print(core.available_devices)

['CPU', 'GPU', 'NPU']


In [1]:
# test loading source and chunking it
from system.tools.load_source import load_source
from system.tools.clean_md import clean_markdown_chunk
from system.tools.chunk_md import chunk_article
source_text = load_source(file_path="C:\\Users\\matse\\sens-website-hugo\\content\\_index.en.md") # change to actual file-path later.
cleaned_text = clean_markdown_chunk(source_text)
print(cleaned_text)  # print first 500 chars of cleaned text

"""
#chunks = chunk_article(source_text)

for i, chunk in enumerate(chunks):
    print(f"--- Chunk {i+1}/{len(chunks)} ---")
    print(chunk)  # print first 500 chars of each chunk
    print()
"""

---
title: "SENS Innovation ApS – Discrete Wearable Activity Sensor"
description: "Better Treatment. Better Research."
draft: false
layout: "custom"
---

{{< hero
  image="/img/bg-front-walking.jpg"
  image_top="/img/sens-motion-logo-white.png"
  image_alt="SENS motion logo"
  title="Discrete Wearable Activity Sensor for Healthcare and Research"
  text="Better Treatment. Better Research."
>}}

{{< textimage
    title="What is SENS motion^(®)"
    subtitle="System Measures:"
    list="Rest time.Standing time.Walking time.Running &amp; High Intensity Movement time.Cycling time.Steps.Intensity.Sleep time and quality"
    image="/img/billede-collage-forside.png"
    alt="SENS motion system overview"
    reverse="false"
    order="body,subtitle,list"
>}}

**SENS motion^(®)** is an integrated system for collecting physical activity data from groups of people. It consists of a wireless activity sensor that automatically transfers data to a secure cloud. It is especially well suited for use in

'\n#chunks = chunk_article(source_text)\n\nfor i, chunk in enumerate(chunks):\n    print(f"--- Chunk {i+1}/{len(chunks)} ---")\n    print(chunk)  # print first 500 chars of each chunk\n    print()\n'

In [ ]:
# categorizing split-types of pages 
textimage_shortcode_split = ["content"]
md_split = [""]

In [21]:
# Test full preprocessing-function by printing the cleaned text and chunked output
import json
from system.tools.load_chunks import load_chunks
#read json-file
json_path = "C:\\Users\\matse\\gig\\src\\system\\content\\chunks.jsonl"

chunks = load_chunks(json_path)

print(len(chunks), "chunks loaded from JSONL")
print(chunks[3]["chunk_text_clean"])

61 chunks loaded from JSONL
title: Thank you for your message

We have received your message and will get back to you as soon as possible
[Back to home page](/)
